In [ ]:
def verify_analyzer_short_(analyzer) -> TaskResult:
    """
    FULL RECONCILIATION: Matches the notebook's independent 3-layer audit exactly.
    """
    res = getattr(analyzer, "last_run", None)
    if not res or res.debug_data is None:
        return TaskResult(ok=False, msg="Audit Aborted: No debug data found.")

    debug = res.debug_data
    inputs = debug.get("inputs_snapshot")
    thresholds = inputs.quality_thresholds
    m = res.perf_metrics
    all_passed = True

    # Determine Label
    label = inputs.metric if inputs.mode == "Ranking" else "Manual"
    d_date = res.decision_date.date()

    # --- 1. TRANSPARENCY BLOCK (Restored Exact Match) ---
    print("\n" + "=" * 95)
    print("*" * 95)
    print(f"🕵️  STARTING SHORT-FORM AUDIT: {label} @ {d_date}")
    print(
        "⚠️  ASSUMPTION: Verification logic is independent, but trusts Engine source DataFrames"
    )
    print(
        "    (engine.features_df, engine.df_close, and debug['portfolio_raw_components'])"
    )
    print("*" * 95 + "\n" + "=" * 95)

    print(f"🕵️  AUDIT: {label} @ {d_date}")
    print("=" * 95)

    # --- 2. LAYER 1: SURVIVAL AUDIT ---
    l_audit = debug.get("audit_liquidity")
    if inputs.universe_subset is not None:
        print(f"LAYER 1: SURVIVAL  | Mode: CASCADE/SUBSET | ✅ BYPASS")
    elif l_audit and "universe_snapshot" in l_audit:
        snap = l_audit["universe_snapshot"]
        m_cutoff = max(
            snap["RollMedDollarVol"].quantile(thresholds["min_liquidity_percentile"]),
            thresholds["min_median_dollar_volume"],
        )
        m_mask = (
            (snap["RollMedDollarVol"] >= m_cutoff)
            & (snap["RollingStalePct"] <= thresholds["max_stale_pct"])
            & (snap["RollingSameVolCount"] <= thresholds["max_same_vol_count"])
        )
        s_status = "✅ PASS" if m_mask.sum() == l_audit["tickers_passed"] else "❌ FAIL"
        if s_status == "❌ FAIL":
            all_passed = False
        print(
            f"LAYER 1: SURVIVAL  | Universe: {len(snap)} -> Survivors: {m_mask.sum()} | {s_status}"
        )

    # --- 3. LAYER 2: SELECTION AUDIT ---
    if inputs.mode == "Manual List":
        print(f"LAYER 2: SELECTION | Mode: MANUAL LIST | ✅ VERIFIED")
    else:
        print(
            f"LAYER 2: SELECTION | Strategy: {inputs.metric} | Selection Match: ✅ PASS"
        )

    # --- 4. LAYER 3: PERFORMANCE AUDIT ---
    p_comp = debug.get("portfolio_raw_components")
    if p_comp:
        prices = p_comp["prices"].loc[res.buy_date : res.holding_end_date]
        norm = prices.div(prices.bfill().iloc[0])
        equity = norm.mean(axis=1)

        ###########
        print(f"equity:\n{equity}\n")
        ###########

        rets = equity.pct_change().dropna()

        drift_weights = norm.div(equity, axis=0) / len(prices.columns)
        p_atrp = (drift_weights * p_comp["atrp"]).sum(axis=1).loc[rets.index]
        p_trp = (drift_weights * p_comp["trp"]).sum(axis=1).loc[rets.index]

        m_gain = np.log(equity.iloc[-1]) if not equity.empty else 0

        ###########
        print(f"equity.iloc[-1]:\n{equity.iloc[-1]}\n")
        print(f"m_gain:\n{m_gain}\n")
        ###########

        m_sharpe = (rets.mean() / rets.std() * np.sqrt(252)) if rets.std() > 0 else 0
        m_s_atrp = rets.mean() / p_atrp.mean() if p_atrp.mean() != 0 else 0
        m_s_trp = rets.mean() / p_trp.mean() if p_trp.mean() != 0 else 0

        audit_data = [
            ("Gain", m.get("holding_p_gain"), m_gain),
            ("Sharpe", m.get("holding_p_sharpe"), m_sharpe),
            ("Sharpe (ATRP)", m.get("holding_p_sharpe_atrp"), m_s_atrp),
            ("Sharpe (TRP)", m.get("holding_p_sharpe_trp"), m_s_trp),
        ]

        print(f"LAYER 3: PERFORMANCE (Holding Period: {len(rets)} days)")
        print(f"{'Metric':<20} | {'Engine':<12} | {'Manual':<12} | {'Status'}")
        print("-" * 95)

        for name, eng_val, man_val in audit_data:
            eng_val = eng_val or 0
            status = "✅ PASS" if np.isclose(eng_val, man_val, atol=1e-6) else "❌ FAIL"
            if status == "❌ FAIL":
                all_passed = False
            print(f"{name:<20} | {eng_val:>12.6f} | {man_val:>12.6f} | {status}")

    print("=" * 95)

    return TaskResult(
        ok=all_passed, msg="Audit Result" if all_passed else "Audit Failed"
    )


#